In [2]:
from pathlib import Path
import sys
import pandas as pd
import tqdm
import json
from concurrent.futures import ThreadPoolExecutor, as_completed

sys.path.insert(0, str(Path('..').resolve())) # Add parent directory to path for imports

from agents import SingleShotAgent, MultiShotAgent, ZeroShotAgent, ImageEditAgent, CriticEditorAgent
from lib.render import render_image
from lib.types import Spec
from lib.ai import gemini_score_aesthetic, gemini_score_edit

specs_dir = Path('../datasets/specs/') # Load canva specs directory
spec_paths = sorted(list(specs_dir.glob('*/spec.json'))) # Get all spec directories
specs_df = pd.DataFrame({
    'template_id': [p.parent.name for p in spec_paths],
    'spec_path': spec_paths
})

print(f"Found {len(specs_df)} design specs")
specs_df.head()

Found 100 design specs


,template_id,spec_path
0,1067w-IQprojiENUA,../datasets/specs/1067w-IQprojiENUA/spec.json
1,1131w-02GXIIidf_4,../datasets/specs/1131w-02GXIIidf_4/spec.json
2,1131w-120Kpwxx3eY,../datasets/specs/1131w-120Kpwxx3eY/spec.json
3,1131w-50t2MzJLmsM,../datasets/specs/1131w-50t2MzJLmsM/spec.json
4,1131w-9cN5biKALLQ,../datasets/specs/1131w-9cN5biKALLQ/spec.json


In [ ]:
AGENT_MODEL = "gemini/gemini-2.5-pro"

specs_dir = Path('../datasets/specs')
edits_dir = Path('../edits')

def process_single_edit(row, agent_type):
    """Process a single edit: agent edit (handles assets & rendering) -> evaluate."""
    
    template_id = row['template_id']
    edit_type_id = row['edit_type_id']
    edit_type = row['edit_type']
    instruction = row['instruction']
    
    # Construct paths
    spec_path = specs_dir / template_id / 'spec.json'
    agent_id = agent_type.lower().replace("agent", "")
    edit_id = f"{template_id}_t{edit_type_id}_{agent_id}"
    edit_dir = edits_dir / edit_id
    output_path = edit_dir / 'spec.json'
    
    # Clean any previous output and re-run
    if edit_dir.exists():
        import shutil
        shutil.rmtree(edit_dir, ignore_errors=True)
    
    # Create agent based on type
    if agent_type == "ZeroShotAgent":
        agent = ZeroShotAgent(model=AGENT_MODEL, verbose=False)
    elif agent_type == "SingleShotAgent":
        agent = SingleShotAgent(model=AGENT_MODEL, image_editor="gpt-image-1", verbose=False)
    elif agent_type == "MultiShotAgent":
        agent = MultiShotAgent(model=AGENT_MODEL, verbose=False)
    elif agent_type == "ImageEditAgent":
        agent = ImageEditAgent(model="gpt-image-1", verbose=False)
    elif agent_type == "CriticEditorAgent":
        agent = CriticEditorAgent(model=AGENT_MODEL, verbose=False, save_renders=True)
    else:
        raise ValueError(f"Unknown agent type: {agent_type}")
    
    # Agent handles EVERYTHING: copies assets, edits, renders
    # print(f"→ Processing {edit_id}...")
    agent.edit(spec_path=spec_path, instruction=instruction, output_path=output_path)
    
    # Evaluate aesthetic score
    render_path = edit_dir / 'render.png'
    original_render_path = spec_path.parent / 'render.png'
    aesthetic_score = gemini_score_aesthetic(render_path) if render_path.exists() else None
    edit_score = (
        gemini_score_edit(original_render_path, render_path, instruction)
        if (original_render_path.exists() and render_path.exists())
        else None
    )
    
    print(f"✓ {edit_id} (aesthetic: {aesthetic_score}, edit: {edit_score})")
    
    return {
        'design': template_id,
        'template_id': edit_type_id,
        'template_name': edit_type,
        'categories': row['categories'],
        'instruction': instruction,
        'agent': agent_id,
        'model': AGENT_MODEL,
        'edit_path': str(output_path),
        'render_path': str(render_path),
        'aesthetic_score': aesthetic_score,
        'edit_score': edit_score,
    }
    

## Test transparent layer editing

In [4]:
import asyncio

FASHIONABLE_WARDROBE_TEMPLATE_ID = "1067w-IQprojiENUA"
LOST_CAT_TEMPLATE_ID = "1131w-02GXIIidf_4"
BRANDING_DESIGN_TEMPLATE_ID = "1600w-GFKaeiEL8R0"
SVG_HEADPHONES_TEMPLATE_ID = "848w-a7AVmpGZf5E"

# Load spec
spec_path = specs_dir / FASHIONABLE_WARDROBE_TEMPLATE_ID / 'spec.json'
spec = json.load(open(spec_path))

r1 = {
    'agent_type': 'SingleShotAgent',
    'template_id': FASHIONABLE_WARDROBE_TEMPLATE_ID,
    'edit_type_id': "tl1",
    'edit_type': 'transparent_layer',
    'instruction': 'Turn the striped shirt blue',
    'categories': ['local', 'image'],
}

r2 = {
    'agent_type': 'CriticEditorAgent',
    'template_id': LOST_CAT_TEMPLATE_ID,
    'edit_type_id': "tl1",
    'edit_type': 'copy_change',
    'instruction': 'Make it a lost dog poster. The dogs name is Greg, and he is a 5yo male Schnauzer.',
    'categories': ['global', 'image', 'text'],
}

r3 = {
    'agent_type': 'CriticEditorAgent',
    'template_id': BRANDING_DESIGN_TEMPLATE_ID,
    'edit_type_id': "tl1",
    'edit_type': 'color_inversion',
    'instruction': 'Turn the poster light themed',
    'categories': ['global', 'image', 'color'],
}

r4 = {
    'agent_type': 'CriticEditorAgent',
    'template_id': SVG_HEADPHONES_TEMPLATE_ID,
    'edit_type_id': "tl1",
    'edit_type': 'svg_updated',
    'instruction': 'Turn the black shape at the bottom a beautiful cobalt blue',
    'categories': ['local', 'image', 'color'],
}
def _run_blocking(func, *args, **kwargs):
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return func(*args, **kwargs)
    else:
        # In Jupyter (running event loop), run in a separate thread
        with ThreadPoolExecutor(max_workers=1) as ex:
            return ex.submit(func, *args, **kwargs).result()

# rest = _run_blocking(process_single_edit, r1, "SingleShotAgent")


In [6]:
# rest = _run_blocking(process_single_edit, r1, "MultiShotAgent")
# rest = _run_blocking(process_single_edit, r1, "CriticEditorAgent")
# rest = _run_blocking(process_single_edit, r2, "CriticEditorAgent")
# rest = _run_blocking(process_single_edit, r3, "CriticEditorAgent")
rest = _run_blocking(process_single_edit, r4, "CriticEditorAgent")



Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



BadRequestError: litellm.BadRequestError: GeminiException BadRequestError - {
  "error": {
    "code": 400,
    "message": "Unable to process input image. Please retry or report in https://developers.generativeai.google/guide/troubleshooting",
    "status": "INVALID_ARGUMENT"
  }
}
